In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from skimage.feature import hog, local_binary_pattern
import numpy as np
import joblib, os
import gc
from load_data import load_for_logistic_regression

Shared preprocessing

In [ ]:
X_train, X_test, y_train, y_test, class_names = load_for_logistic_regression()

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=500)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

def extract_hog(X):
    features = []
    for row in X:
        img = row.reshape(64, 64)
        h = hog(img, orientations=12, pixels_per_cell=(16, 16), cells_per_block=(1, 1), feature_vector=True)
        features.append(h)
    return np.array(features)

def extract_lbp(X, radius=3, n_points=24, bins=32):
    features = []
    for row in X:
        img = row.reshape(64, 64)
        lbp = local_binary_pattern(img, n_points, radius, method='uniform')
        hist, _ = np.histogram(lbp, bins=bins, range=(0, n_points + 2), density=True)
        features.append(hist)
    return np.array(features)

X_train_lbp = extract_lbp(X_train)
X_test_lbp = extract_lbp(X_test)
X_train_hog = extract_hog(X_train)
X_test_hog = extract_hog(X_test)

del X_train_scaled, X_test_scaled
gc.collect()

The model below is our baseline, raw linear regression.

In [ ]:
basic = LogisticRegression(max_iter=1000, random_state=42,n_jobs=-1)
basic.fit(X_train, y_train)

y_pred = basic.predict(X_test)
os.makedirs('Reports', exist_ok=True)
with open('Reports/basic_report.txt','w') as f:
    f.write(str(accuracy_score(y_test,y_pred)))
    f.write(classification_report(y_test,y_pred,target_names=class_names))
os.makedirs('Weights/basic', exist_ok=True)
joblib.dump(basic, 'Weights/basic/model.joblib')
del X_train, X_test
gc.collect()

27

Next we have a step up, using an sklearn pipeline the chain together several enhancements over the basic model:
- StandardScaler for more uniform penalties during training
- Principled Component Analysis to turn the pixels into somewhat comprehensible data for the model (e.g. it could pick up on brightness or texture instead of going off of raw recognition) and also speed up training (150 vs 64*64 features).
- saga + multinomial to have one model that considers all classes as a possibility vs 131 models 

In [5]:
medium = LogisticRegression(solver='lbfgs', multi_class='multinomial', max_iter=500, n_jobs=-1,random_state=42)

In [ ]:
medium = LogisticRegression(solver='lbfgs', multi_class='multinomial', max_iter=500, n_jobs=-1,random_state=42)
medium.fit(X_train_pca, y_train)
y_pred = medium.predict(X_test_pca)
os.makedirs('Reports', exist_ok=True)
with open('Reports/medium_report.txt','w') as f:
    f.write(str(accuracy_score(y_test,y_pred)))
    f.write(classification_report(y_test,y_pred,target_names=class_names))

os.makedirs('Weights/medium', exist_ok=True)
joblib.dump(medium, 'Weights/medium/model.joblib')

/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


['weights/medium/model.joblib']

Next we have high, which on top of everything in medium, implements HOG or Histogram of Oriented Gradients features.

While the PCA picks up on visual characteristics, it does not recognize shape or edges. HOG compliments PCA by enabling shape and edge detection by splitting the image into cells and calculating the gradient direction and magnitude for each pixel. It then creates a histogram of bins each bin covering a portion of arc of edge direction. Cells are then grouped and overlapped to reduce variance and noise.

In this case, to create a feature stack I flattenned the HOG so it can be paired with the PCA from before.

In [7]:
X_train_high = np.hstack([X_train_pca, X_train_hog])
X_test_high = np.hstack([X_test_pca, X_test_hog])

high = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(solver='lbfgs', multi_class='multinomial', max_iter=500, n_jobs=-1, random_state=42))
])

In [ ]:
high.fit(X_train_high, y_train)
y_pred = high.predict(X_test_high)
os.makedirs('Reports', exist_ok=True)
with open('Reports/high_report.txt','w') as f:
    f.write(str(accuracy_score(y_test,y_pred)))
    f.write(classification_report(y_test,y_pred,target_names=class_names))

os.makedirs('Weights/high', exist_ok=True)
joblib.dump({
    'pipeline': high,
    'scaler': scaler,
    'pca': pca,
}, 'Weights/high/model.joblib')

del X_train_high, X_test_high
gc.collect()

/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


54

Finally, we have the max model. We introduce color histograms hence the reload with grayscale=False. The color histogram creates a map of what colors are present, but not where they are. For fruits with a very distinct color profile, it is highly effective for classification. With great features comes great fitting, sometimes too great. While seevral C values were tested, the default of 1.0 remains the best, although the difference was less than 2% accuracy across 0.1-5.0.

In [9]:
from load_data import build_dataset_index, iter_image_paths, load_single_image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

def extract_color_histograms_streaming(split, bins=32):
    index = build_dataset_index()
    samples = list(iter_image_paths(index, split))
    out = np.empty((len(samples), bins * 3), dtype=np.float32)
    for i, (path, _) in enumerate(samples):
        img = load_single_image(path, image_size=(64, 64), grayscale=False, flatten=False, dtype=np.uint8)
        img_f = img.astype(np.float32) / 255.0
        out[i, :bins] = np.histogram(img_f[:, :, 0], bins=bins, range=(0, 1))[0] / (64 * 64)
        out[i, bins:2*bins] = np.histogram(img_f[:, :, 1], bins=bins, range=(0, 1))[0] / (64 * 64)
        out[i, 2*bins:] = np.histogram(img_f[:, :, 2], bins=bins, range=(0, 1))[0] / (64 * 64)
    return out

X_train_color = extract_color_histograms_streaming("train")
X_test_color = extract_color_histograms_streaming("test")

X_train_max = np.hstack([X_train_pca, X_train_hog, X_train_color, X_train_lbp])
X_test_max = np.hstack([X_test_pca, X_test_hog, X_test_color, X_test_lbp])

del X_train_hog, X_test_hog, X_train_color, X_test_color, X_train_lbp, X_test_lbp
gc.collect()

0

In [ ]:
max_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(solver='lbfgs', multi_class='multinomial', C=1.0, max_iter=500, n_jobs=-1, random_state=42))
])

max_model.fit(X_train_max, y_train)
y_pred = max_model.predict(X_test_max)
os.makedirs('Reports', exist_ok=True)
with open('Reports/max_report.txt','w') as f:
    f.write(str(accuracy_score(y_test,y_pred)))
    f.write(classification_report(y_test,y_pred,target_names=class_names))
os.makedirs('Weights/max', exist_ok=True)
joblib.dump({
    'pipeline': max_model,
    'scaler': scaler,
    'pca': pca,
}, 'Weights/max/model.joblib')

del X_train_max, X_test_max
gc.collect()

/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


27